# ETAP AI Platform — Interactive Load Flow Demo

This notebook demonstrates how to perform a **Load Flow Analysis** using the ETAP AI Engineering Platform API. Load flow (power flow) analysis is the cornerstone of power system studies, determining voltage magnitudes, phase angles, and power flows throughout the network.

## What You'll Learn
- How to define a power system model (buses, lines, generators, loads)
- How to submit a load flow study to the ETAP AI API
- How to interpret results including voltage profiles and line loading
- How to validate results against IEEE 141/399 standards

## Prerequisites
- Access to the ETAP AI Engineering Platform (running on localhost:8000 or remote)
- A valid API key or JWT token
- Python 3.10+ with `requests` and `matplotlib` installed

In [ ]:
# Cell 1: Setup and Configuration
import matplotlib.pyplot as plt
import requests

# Configuration — adjust these for your environment
BASE_URL = "http://localhost:8000/api/v1"
API_KEY = "your-api-key-here"  # Replace with your actual API key

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print("Configuration loaded.")
print(f"API Base URL: {BASE_URL}")

## Step 1: Define the Power System Model

We'll create a simple 5-bus industrial power system model. This represents a typical facility with a utility connection (slack bus), two generator buses, and two load buses connected through transmission lines and transformers.

The system is defined per the ETAP AI Platform's `SystemSpec` schema, which uses per-unit values on a common MVA base.

In [ ]:
# Cell 2: Define the 5-Bus Power System
system_data = {
    "base_mva": 100.0,
    "buses": [
        {
            "bus_id": 1,
            "voltage_magnitude": 1.05,
            "voltage_angle": 0.0,
            "bus_type": "slack",
            "base_kv": 138.0
        },
        {
            "bus_id": 2,
            "voltage_magnitude": 1.0,
            "bus_type": "pq",
            "load_power_real": 45.0,
            "load_power_imag": 15.0,
            "base_kv": 13.8
        },
        {
            "bus_id": 3,
            "voltage_magnitude": 1.02,
            "bus_type": "pv",
            "generation_power_real": 30.0,
            "base_kv": 13.8,
            "q_min": -30.0,
            "q_max": 30.0,
            "voltage_setpoint": 1.02
        },
        {
            "bus_id": 4,
            "voltage_magnitude": 1.0,
            "bus_type": "pq",
            "load_power_real": 40.0,
            "load_power_imag": 20.0,
            "base_kv": 4.16
        },
        {
            "bus_id": 5,
            "voltage_magnitude": 1.0,
            "bus_type": "pq",
            "load_power_real": 25.0,
            "load_power_imag": 10.0,
            "base_kv": 0.48
        }
    ],
    "lines": [
        {
            "line_id": 1,
            "from_bus_id": 1,
            "to_bus_id": 2,
            "r1": 0.01,
            "x1": 0.05,
            "bshunt1": 0.02,
            "rating_mva": 100.0
        },
        {
            "line_id": 2,
            "from_bus_id": 2,
            "to_bus_id": 3,
            "r1": 0.02,
            "x1": 0.08,
            "bshunt1": 0.03,
            "rating_mva": 80.0
        },
        {
            "line_id": 3,
            "from_bus_id": 2,
            "to_bus_id": 4,
            "r1": 0.005,
            "x1": 0.04,
            "bshunt1": 0.01,
            "rating_mva": 60.0
        },
        {
            "line_id": 4,
            "from_bus_id": 4,
            "to_bus_id": 5,
            "r1": 0.003,
            "x1": 0.03,
            "bshunt1": 0.005,
            "rating_mva": 30.0
        }
    ],
    "transformers": [
        {
            "transformer_id": 1,
            "from_bus_id": 1,
            "to_bus_id": 2,
            "r1": 0.005,
            "x1": 0.05,
            "tap_ratio": 1.0,
            "phase_shift_deg": 0.0
        }
    ],
    "generators": [
        {
            "generator_id": 1,
            "bus_id": 1,
            "x1": 0.2,
            "internal_voltage_mag": 1.05
        },
        {
            "generator_id": 2,
            "bus_id": 3,
            "x1": 0.25,
            "internal_voltage_mag": 1.02,
            "power_real": 30.0
        }
    ],
    "loads": [
        {"load_id": 1, "bus_id": 2, "p_mw": 45.0, "q_mvar": 15.0},
        {"load_id": 2, "bus_id": 4, "p_mw": 40.0, "q_mvar": 20.0},
        {"load_id": 3, "bus_id": 5, "p_mw": 25.0, "q_mvar": 10.0}
    ]
}

print(f"System defined: {len(system_data['buses'])} buses, {len(system_data['lines'])} lines, "
      f"{len(system_data['generators'])} generators, {len(system_data['loads'])} loads")

## Step 2: Validate the System Model

Before running a study, it's good practice to validate the system model for data integrity, connectivity, and basic feasibility. The `/api/v1/system/validate` endpoint checks for common errors like missing slack buses, disconnected islands, and invalid parameter ranges.

In [ ]:
# Cell 3: Validate the System
try:
    response = requests.post(
        f"{BASE_URL}/system/validate",
        json=system_data,
        headers=HEADERS,
        timeout=30
    )
    validation = response.json()

    if validation.get("valid"):
        print("✅ System model is valid!")
        print(f"   Buses: {validation['statistics']['bus_count']}")
        print(f"   Lines: {validation['statistics']['line_count']}")
        print(f"   Has Slack Bus: {validation['statistics']['has_slack_bus']}")
        print(f"   Connected: {validation['statistics']['connected']}")
    else:
        print("❌ System model has errors:")
        for err in validation.get("errors", []):
            print(f"   - {err}")
except requests.exceptions.ConnectionError:
    print("⚠️  Cannot connect to the API. Make sure the engineering service is running.")
    print("   Run: python engineering_service.py --host 0.0.0.0 --port 8000")

## Step 3: Run the Load Flow Study

Now we submit the load flow study using the Newton-Raphson method, which is the most common and robust method for power flow analysis. The API supports three methods:
- `newton_raphson` — Full NR (most robust, handles all system types)
- `fast_decoupled` — FD (faster, good for well-conditioned systems)
- `dc_power_flow` — Linearized DC approximation (fastest, less accurate)

In [ ]:
# Cell 4: Run Load Flow
study_request = {
    "study_type": "load_flow",
    "system": system_data,
    "parameters": {
        "method": "newton_raphson",
        "max_iterations": 50,
        "tolerance": 1e-6
    }
}

try:
    response = requests.post(
        f"{BASE_URL}/studies/run",
        json=study_request,
        headers=HEADERS,
        timeout=60
    )
    result = response.json()

    print(f"Study ID: {result.get('task_id', 'N/A')}")
    print(f"Status: {result.get('status', 'N/A')}")
    print(f"Converged: {result.get('converged', 'N/A')}")
    print(f"Iterations: {result.get('iterations', 'N/A')}")
    print(f"Execution Time: {result.get('execution_time_ms', 'N/A'):.1f} ms")
except requests.exceptions.ConnectionError:
    print("⚠️  Cannot connect to the API.")
    # Provide sample results for demonstration
    result = {
        "converged": True,
        "iterations": 5,
        "results": {
            "buses": {
                "1": {"voltage_magnitude_pu": 1.05, "voltage_angle_deg": 0.0, "power_generated_mw": 82.5, "power_generated_mvar": 25.3},
                "2": {"voltage_magnitude_pu": 0.982, "voltage_angle_deg": -5.23, "power_consumed_mw": 45.0, "power_consumed_mvar": 15.0},
                "3": {"voltage_magnitude_pu": 1.02, "voltage_angle_deg": -3.12, "power_generated_mw": 30.0, "power_generated_mvar": 8.5},
                "4": {"voltage_magnitude_pu": 0.965, "voltage_angle_deg": -8.45, "power_consumed_mw": 40.0, "power_consumed_mvar": 20.0},
                "5": {"voltage_magnitude_pu": 0.941, "voltage_angle_deg": -12.3, "power_consumed_mw": 25.0, "power_consumed_mvar": 10.0}
            },
            "losses_mw": 2.5
        },
        "validation": {"all_voltages_within_limits": False, "all_lines_thermal_ok": True}
    }
    print("(Using sample results for demonstration)")

## Step 4: Analyze Results — Voltage Profile

The voltage profile is one of the most important outputs of a load flow study. IEEE 141 and IEEE 399 recommend that bus voltages remain within ±5% of nominal (0.95–1.05 pu) for normal operating conditions. Let's visualize the voltage at each bus.

In [ ]:
# Cell 5: Plot Voltage Profile
bus_results = result.get("results", {}).get("buses", {})

bus_ids = sorted(bus_results.keys(), key=int)
voltages = [bus_results[bid].get("voltage_magnitude_pu", 0) for bid in bus_ids]
angles = [bus_results[bid].get("voltage_angle_deg", 0) for bid in bus_ids]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Voltage Magnitude
colors = ['green' if 0.95 <= v <= 1.05 else 'red' for v in voltages]
ax1.bar(bus_ids, voltages, color=colors, alpha=0.7, edgecolor='black')
ax1.axhline(y=1.05, color='orange', linestyle='--', linewidth=1.5, label='Upper Limit (1.05 pu)')
ax1.axhline(y=0.95, color='orange', linestyle='--', linewidth=1.5, label='Lower Limit (0.95 pu)')
ax1.axhline(y=1.0, color='gray', linestyle=':', linewidth=1, label='Nominal (1.0 pu)')
ax1.set_ylabel('Voltage Magnitude (pu)')
ax1.set_title('Load Flow Results — Voltage Profile')
ax1.legend(loc='lower left', fontsize=8)
ax1.set_ylim(0.9, 1.1)
ax1.grid(True, alpha=0.3)

# Voltage Angle
ax2.bar(bus_ids, angles, color='steelblue', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Bus ID')
ax2.set_ylabel('Voltage Angle (degrees)')
ax2.set_title('Voltage Angle Profile')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('load_flow_voltage_profile.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify problem buses
problem_buses = [bid for bid, v in zip(bus_ids, voltages) if not (0.95 <= v <= 1.05)]
if problem_buses:
    print(f"\n⚠️  Buses outside voltage limits: {', '.join(problem_buses)}")
    print("   Recommendation: Consider reactive power compensation or tap changes")
else:
    print("\n✅ All buses within voltage limits (0.95–1.05 pu)")

## Step 5: IEEE Compliance Check

The ETAP AI Platform automatically validates results against IEEE standards. Let's review the compliance status and identify any violations that need corrective action.

In [ ]:
# Cell 6: Compliance Summary
validation = result.get("validation", {})
losses = result.get("results", {}).get("losses_mw", 0)
total_load = sum(
    bus_results[bid].get("power_consumed_mw", 0)
    for bid in bus_ids
)

print("====== IEEE Compliance Report =====")
print("\n📊 System Summary:")
print(f"   Total Load:        {total_load:.1f} MW")
print(f"   Total Losses:      {losses:.2f} MW")
print(f"   Loss Percentage:   {(losses/max(total_load,1))*100:.2f}%")
print("\n✅ IEEE 141/399 Checks:")
print(f"   Voltages within limits:   {'✅ Yes' if validation.get('all_voltages_within_limits') else '❌ No'}")
print(f"   Lines within thermal:     {'✅ Yes' if validation.get('all_lines_thermal_ok') else '❌ No'}")
print(f"   Power balance error:      {validation.get('power_balance_error_mw', 'N/A')} MW")
print("\n📋 Bus Voltage Details:")
for bid in bus_ids:
    vm = bus_results[bid].get('voltage_magnitude_pu', 0)
    status = '✅' if 0.95 <= vm <= 1.05 else '❌'
    print(f"   Bus {bid}: {vm:.3f} pu {status}")

## Step 6: Try Different Methods

You can compare results from different load flow methods. The Newton-Raphson method is the most accurate but computationally intensive. The Fast Decoupled method is faster but may not converge for all systems. Try modifying the `method` parameter to see the differences.

```python
# Try Fast Decoupled method
study_request['parameters']['method'] = 'fast_decoupled'
response = requests.post(f"{BASE_URL}/studies/run", json=study_request, headers=HEADERS)
fd_result = response.json()
```

## Summary

In this demo, you learned how to:
1. Define a power system model using the ETAP AI Platform schema
2. Validate the model before running studies
3. Run a Newton-Raphson load flow analysis
4. Visualize and interpret voltage profiles
5. Check IEEE 141/399 compliance

For more information, see the [API Reference](../API_REFERENCE.md) and [Compliance Matrix](../COMPLIANCE.md).